# 05 — Conclusiones

Este notebook cierra el proyecto respondiendo las preguntas definidas en la inspección inicial,
diferenciando **evidencia** (lo observado), **interpretación** (qué significa) y **conclusión**
(qué se responde). Es coherente con el informe final (`reports/informe_final.pdf`) y la aplicación
Streamlit.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/streaming_users_clean.csv", parse_dates=["last_login_date"])
log = pd.read_csv("../logs/pipeline_log.csv")
print(f"Dataset final: {df.shape[0]} filas x {df.shape[1]} columnas "
      f"(retención {log['Retención (%)'].iloc[-1]} %)")
log.tail(3)

Dataset final: 8000 filas x 9 columnas (retención 98.04 %)


,Paso,Descripción,Filas,Nulos,Retención (%)
8,8,"tickets inválidos (-1, 99, 150: 96 casos) -> i...",8000,320,98.04
9,9,Fechas: parseo de 2 formatos; inválidas y futu...,8000,610,98.04
10,10,Derivación de days_since_last_login (recencia ...,8000,1220,98.04


## Respuestas a las preguntas de análisis

**P1 — Distribución del consumo.**
*Evidencia:* mediana ~770 min/mes, asimetría 2,4, cola hasta ~4.200 min.
*Conclusión:* el usuario típico mira ~13 horas al mes, pero la media está inflada por una minoría
intensiva; toda métrica de consumo del negocio debería reportarse con medianas.

**P2 — Consumo por plan (hallazgo principal).**
*Evidencia:* medianas de 553 / 840 / 1.127 min para Básico / Estándar / Premium, patrón repetido en los
7 países.
*Conclusión:* el plan es el mejor diferenciador del nivel de uso: un Premium típico consume el doble que
un Básico. Dado que Básico concentra el 45 % de la base, hay margen para estrategias de upgrade
orientadas a los básicos de alto consumo.

**P3 — Edad y consumo.**
*Evidencia:* r ≈ 0,006, nube sin tendencia.
*Conclusión:* la edad no explica la intensidad de uso; no conviene segmentar campañas de consumo por edad.

**P4 — Género favorito por país.**
*Evidencia:* cada género representa 12-16 % en todos los países.
*Conclusión:* las preferencias son homogéneas en la región; el catálogo no requiere diferenciación por
mercado según estos datos.

**P5 — Reducción de dimensionalidad.**
*Evidencia:* varianza explicada casi uniforme (~25 % por componente); 3 de 4 componentes para llegar
al 75 %; correlaciones |r| < 0,02.
*Conclusión:* las variables numéricas son independientes entre sí y PCA no logra comprimirlas; la
estructura del dataset está en las categóricas (plan), no en combinaciones lineales de las numéricas.

## Calidad de datos: qué se hizo y qué efecto tuvo

- Retención del **98 %** (8.160 → 8.000 filas); solo se eliminaron duplicados.
- Valores imposibles (edades -5/150, minutos negativos, centinelas 99999 y -1/99/150) convertidos a
  faltante e imputados con medianas (por plan en el caso del consumo).
- 3 categóricas normalizadas (69 variantes de escritura → 17 categorías reales).
- Fechas: 2 formatos unificados; inválidas y futuras a `NaT` (7,6 % final) sin eliminar filas.
- Todo el proceso quedó trazado en `logs/pipeline_log.csv` con el esquema evidencia → decisión → impacto.

## Limitaciones

- El alcance de las conclusiones está condicionado por la información disponible y por las decisiones
  documentadas durante el proceso (imputaciones y recategorizaciones incluidas).
- El dataset es un corte transversal sin dimensión temporal del consumo: no permite analizar evolución,
  retención ni causalidad (por ejemplo, si el plan causa mayor consumo o los usuarios intensivos eligen
  planes superiores).
- Un 7,6 % de los usuarios queda sin fecha de último acceso válida, lo que reduce la muestra del PCA y
  de cualquier análisis de recencia.
- La variable `favorite_genre` declara una preferencia única por usuario y no refleja el consumo real
  por género.

## Mejoras futuras

- Incorporar información adicional (historial de visualizaciones, ingresos por usuario, fecha de alta)
  que permita ampliar el alcance del análisis hacia retención y valor del cliente.
- Repetir el análisis con cortes periódicos del dataset para estudiar evolución temporal.
- Explorar técnicas de segmentación que combinen variables categóricas y numéricas (por ejemplo,
  clustering con distancias mixtas), dado que PCA sobre las numéricas no encontró estructura.